# Part 1 — Extract dimensions from CubiCasa5k and annotate the PNGs**What this notebook does.** Each plan folder holds a `model.svg` and one to three PNGrenders. The PNGs show the floor plan but carry no dimensions. The SVG carries them inhidden text: every room has a `DimensionMeasureLabel` group with `style="display: none"`holding a string like `2.84 m x 1.58 m`. This notebook reads those, verifies them againstthe drawn geometry, and burns them onto a copy of each PNG.**Two things confirmed by inspecting a real file:**1. The drawing scale is **100 SVG units per metre** — one unit is one centimetre.2. The dimensions are already written in the file. We read them rather than deriving them,   then cross-check against the polygons so a bad file gets caught instead of producing   plausible-looking wrong numbers.

## 1. Install and importOne import per line so it is clear what each is for.

In [ ]:
# Colab already has Pillow and numpy. lxml is the XML parser we use for the SVG.!pip install -q lxml

In [ ]:
import os                      # building and joining file pathsimport json                    # writing the manifest of extracted dimensionsimport shutil                  # copying the original files into the output folderimport zipfile                 # packaging the finished output as one downloadimport re                      # pulling "2.84 m x 1.58 m" out of the hidden label textimport numpy as np             # finding the black linework in the PNGfrom lxml import etree         # reading the SVG, which is just XMLfrom PIL import Image          # opening and saving the PNGfrom PIL import ImageDraw      # drawing the dimension lines and textfrom PIL import ImageFont      # loading a scalable font for the labelsfrom statistics import median  # combining per-room scale checks into one figure

## 2. Get the pipeline codeThe parsing, alignment and annotation logic lives in `core/`. Keeping it in modules ratherthan pasting it into cells means the notebook and the web app run the exact same code, so afix in one place fixes both.

In [ ]:
# Upload the core/ folder to your Colab session, or clone it from your repo.# The folder must contain: svg_parse.py, align.py, annotate.py, batch.pyimport syssys.path.insert(0, "/content")          # so "from core...." resolvesfrom core.svg_parse import parse_svg    # SVG -> dimensionsfrom core.annotate import annotate      # dimensions -> annotated PNGfrom core.batch import run_batch        # do that for a whole folderfrom core.batch import zip_output       # package the result for download

## 3. Point at the datasetKept in its own cell so you can swap a single plan for the full set without touchinganything else.

In [ ]:
# Mount Drive if the dataset lives there. Skip this if you uploaded it directly.from google.colab import drivedrive.mount("/content/drive")

In [ ]:
DATASET_ROOT = "/content/drive/MyDrive/cubicasa5k"   # folders of plans, each with model.svgOUTPUT_ROOT  = "/content/annotated"                  # where annotated copies are writtenZIP_PATH     = "/content/annotated_plans.zip"        # the single file you download at the endos.makedirs(OUTPUT_ROOT, exist_ok=True)              # create the output folder if it is newprint("Dataset :", DATASET_ROOT)print("Output  :", OUTPUT_ROOT)

## 4. Run one plan firstNever batch 100 files before you have looked at one. This cell prints what was extracted soyou can sanity-check the numbers against the drawing.

In [ ]:
SAMPLE = os.path.join(DATASET_ROOT, "plan_0001")     # change to any plan folder you haveplan = parse_svg(os.path.join(SAMPLE, "model.svg"), "plan_0001")   # read the SVGprint("Scale        :", plan.px_per_m, "px per metre")             # expect ~100print("Scale source :", plan.scale_source)                         # how it was determinedprint("Disagreement : {:.2%}".format(plan.scale_confidence))       # spread across rooms; want < 5%print("Overall      : {:.2f} m x {:.2f} m".format(plan.overall_width_m, plan.overall_depth_m))print("Floor area   : {:.1f} m2".format(plan.internal_area_m2))print()for room in plan.rooms:                                            # one line per room    print("{:<10} {:<16} {:>6.2f} x {:<6.2f} m   {:>6.2f} m2".format(        room.code, room.name, room.width_m, room.depth_m, room.area_m2))

## 5. Annotate one PNG and look at itThe `annotate` call fits the SVG coordinates onto the raster first. The two are not thesame size — the renders are padded — so the scale factor is solved by matching the walllinework rather than assumed.

In [ ]:
annotate(    plan,                                                  # the dimensions we just extracted    os.path.join(SAMPLE, "F1_original.png"),               # the PNG to annotate    "/content/preview.png",                                # where to write the annotated copy    units="m",                                             # "m", "ft", or "both"    style="dimension_lines",                               # or "corner_label" for a title block    room_labels=True,                                      # per-room size and area)from IPython.display import Image as ShowImage             # Colab's inline image viewerShowImage("/content/preview.png", width=1000)              # check it before batching

## 6. Run the full batch`resume=True` means an interrupted session picks up where it stopped — Colab disconnectshappen, and re-running should not redo work. Every plan is wrapped so one broken filecannot kill the run.

In [ ]:
def show_progress(done, total, plan_id, status):           # called after each plan    print("[{}/{}] {} -> {}".format(done, total, plan_id, status))summary = run_batch(    DATASET_ROOT,                                          # where to read from    OUTPUT_ROOT,                                           # where to write to    units="m",                                             # units for the annotations    style="dimension_lines",                               # annotation style    room_labels=True,                                      # draw per-room dimensions    limit=100,                                             # set to None for the whole dataset    resume=True,                                           # skip plans already done    progress=show_progress,                                # print progress as it goes)print()print(summary)                                             # counts of ok / estimated / failed

## 7. Check what came out`status` is `ok` when the scale was confirmed against the room labels, `estimated` when itfell back to the 100 px/m convention, and `failed` when the plan could not be read. Only`ok` plans should feed the accessibility review in Part 2.

In [ ]:
with open(os.path.join(OUTPUT_ROOT, "manifest.json")) as fh:   # the combined results    manifest = json.load(fh)counts = {}                                                    # tally the statusesfor record in manifest:    counts[record["status"]] = counts.get(record["status"], 0) + 1print(counts)for record in manifest:                                        # show anything that needs a look    if record["status"] != "ok":        print(record["plan_id"], record["status"], record.get("error", record.get("warnings")))

In [ ]:
import pandas as pd                                            # for a quick look at the tabletable = pd.read_csv(os.path.join(OUTPUT_ROOT, "dimensions.csv"))   # one row per roomprint(table.shape)table.head(20)

## 8. DownloadThe output folder keeps each plan's original `model.svg` and PNGs next to the annotatedcopies, plus a `dimensions.json` per plan and a combined `manifest.json` and`dimensions.csv` at the top. Everything zips into one file.

In [ ]:
zip_output(OUTPUT_ROOT, ZIP_PATH)                              # package the whole folderprint("{:.1f} MB".format(os.path.getsize(ZIP_PATH) / 1e6))     # check it is a sane sizefrom google.colab import files                                 # Colab's download helperfiles.download(ZIP_PATH)                                       # save it to your machine